# PolicyRec v1.1.9 — region 정규화 추가 (v1.1.8 위)

## 이 노트북의 역할

`raw_v1_1_8.csv` (600건) → 임베딩 단계로 들어갈 수 있는 깨끗한 main_csv 생성.

### 처리 순서

1. **raw 로드 + 정규화** (제목/기관/기간/URL)
2. **카테고리 룰표 v2 적용** → `s_category` 부여
3. **스코프 룰표 v2 적용** → `_scope`, `_scope_reason` 부여
4. **dedupe 처리**
   - 강한 신호 (URL exact) → 자동 other (`duplicate_url`)
   - 완전 동일 (제목+기관+기간) → 자동 other (`duplicate_exact`)
   - 자매 공고 (애매한 케이스) → review 큐
5. **target_tags 부여**
6. **region 정규화** (룰표 `region_rule.csv`, ⭐ v1.1.9 신규)
7. **main_csv 출력** (임베딩용 깨끗한 데이터)
8. **review 큐 csv 출력** (사람 검토용)

## v1.1.8 → v1.1.9 변경점

| 항목 | v1.1.8 | v1.1.9 |
| :--- | :--- | :--- |
| region 컬럼 | raw 단계에서 부처명/기관명이 region에 혼입 (예: 중소벤처기업부 23건, 제주특별자치도경제통상진흥원 경영본부 청년센터 77건) | **룰표(`region_rule.csv`) 적용해 정리** |
| 시도 alias | 서울/서울특별시 혼재 | **긴 공식 표기로 통일** (서울 → 서울특별시 등) |
| 룰표 파일명 패턴 | cat/scope는 노트북 버전 따라감 | **신규 region_rule은 단일 파일** (target_tags 패턴 따름. 룰 변경 시에만 버전 업) |

## (참고) v1 대비 v2 룰표 변경점 — v1.1.8 시점 정리

| 항목 | v1 | v2 |
| :--- | :--- | :--- |
| civic 처리 | 키워드 8개 | 카테고리 룰 (`참여/기반` 자동) + 보강 키워드 5개 |
| 위험 키워드 | 경진대회/박람회/페스티벌/서포터즈 포함 | **모두 제거** (창업지원사업이라 main 유지해야 함) |
| dedupe | review 큐만 분리 | 자동 처리 + review 큐 분리 |
| 중복 카테고리 | 처리 안 됨 | youth `참여･기반,참여･기반` 등 처리

## 0. 셋업

In [ ]:
from pathlib import Path
import re
import html
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(v): print(v)

# ============================================================
# 경로 설정 (팀원 v1.1.4 폴더 구조 기반)
# ============================================================
PROJECT_ROOT = Path.cwd()
VERSION = "v1_1_10"  # 노트북·main 출력 버전. raw/룰표는 아래에서 개별 지정.


CSV_ROOT      = PROJECT_ROOT / "data" / "csv"
CSV_RAW_DIR   = CSV_ROOT / "raw"
CSV_MAIN_DIR  = CSV_ROOT / "main"
CSV_RULE_DIR  = CSV_ROOT / "rule"
CSV_REVIEW_DIR = CSV_ROOT / "review"
for d in [CSV_MAIN_DIR, CSV_RULE_DIR, CSV_REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 입력
# - raw는 v1.1.8 그대로 사용 (raw 자체는 변경 없음)
RAW_CSV       = CSV_RAW_DIR  / "raw_v1_1_8.csv"

# 룰표 (v1.1.9부터 모두 단일 파일 패턴으로 통일)
# 정책: 룰표 파일은 노트북 버전을 따라가지 않는다. 룰이 바뀌면 같은 파일 안에서 수정.
# 변경 이력은 git history로 추적.
# 참고: v1.1.8 시점에 팀원이 만든 cat_rule_v1_1_8.csv / scope_rule_v1_1_8.csv는 아카이브로 그대로 보존됨.
CAT_RULE_CSV    = CSV_RULE_DIR / "cat_rule.csv"
SCOPE_RULE_CSV  = CSV_RULE_DIR / "scope_rule.csv"
REGION_RULE_CSV = CSV_RULE_DIR / "region_rule.csv"
TAGS_RULE_CSV   = CSV_RULE_DIR / "target_tags_rule.csv"

# 출력 (VERSION 따라감 → main_v1_1_9.csv)
MAIN_CSV       = CSV_MAIN_DIR  / f"main_{VERSION}.csv"
REVIEW_CSV     = CSV_REVIEW_DIR / f"review_queue_{VERSION}.csv"

print("[입력]")
print(f"  raw           : {RAW_CSV}")
print(f"  cat_rule      : {CAT_RULE_CSV}")
print(f"  scope_rule    : {SCOPE_RULE_CSV}")
print(f"  region_rule   : {REGION_RULE_CSV}")
print(f"  target_tags   : {TAGS_RULE_CSV}")
print()
print("[출력]")
print(f"  main (임베딩용) : {MAIN_CSV}")
print(f"  review queue   : {REVIEW_CSV}")

## 1. raw csv 로드 + 기본 정리

In [ ]:
df = pd.read_csv(RAW_CSV, dtype=str).fillna("")
print(f"raw shape: {df.shape}")

# HTML entity 풀기 (제목/기관/카테고리) + entity 잔존 처리
entity_rx = r"&#[0-9]+;|&amp;|&lt;|&gt;|&quot;"
for col in ["title", "supervising_agency", "operating_agency", "category", "subcategory"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda v: html.unescape(str(v)).strip())

import re as _re_check
title_entity_after = df["title"].astype(str).str.contains(entity_rx, regex=True, na=False).sum()
print(f"[title HTML entity] 처리 후 잔존: {title_entity_after}건")

# summary 정리: HWP JSON 메타데이터 제거 + HTML 태그 변환
def clean_summary(text):
    """
    summary 텍스트 정리:
    1. HWP JSON 메타데이터 제거
    2. 블록 태그(p, div, br) -> 줄바꿈
    3. 나머지 HTML 태그 제거
    4. HTML entity 디코딩
    5. 다중 공백/줄바꿈 정리
    """
    if not text or not str(text).strip():
        return ""
    
    text = str(text)
    
    # 1. HWP JSON 메타데이터 제거
    text = re.sub(r"<!--\[data-hwpjson\].*?-->", "", text, flags=re.DOTALL)
    
    # 2. 블록 태그를 줄바꿈으로 변환
    NEWLINE = chr(10)
    text = re.sub(r"</?(p|div|br|li|tr)\s*/?>", NEWLINE, text, flags=re.IGNORECASE)
    
    # 3. 나머지 HTML 태그 제거
    text = re.sub(r"<[^>]+>", "", text)
    
    # 4. HTML entity 디코딩
    text = html.unescape(text)
    
    # 5. 다중 공백/줄바꿈 정리
    text = re.sub(NEWLINE + r"\s*" + NEWLINE, NEWLINE, text)
    text = re.sub(r"[ \t]+", " ", text)
    
    return text.strip()

if "summary" in df.columns:
    before_html_count = df["summary"].astype(str).str.contains("<", regex=False).sum()
    df["summary"] = df["summary"].apply(clean_summary)
    after_html_count = df["summary"].astype(str).str.contains("<", regex=False).sum()
    print(f"[summary HTML 정리] 처리 전: {before_html_count}건 → 처리 후: {after_html_count}건")

# provider: supervising_agency 우선, 없으면 operating_agency
df["provider"] = df["supervising_agency"].where(
    df["supervising_agency"].astype(str).str.strip() != "",
    df["operating_agency"]
)

print()
print("=== source 분포 ===")
display(df["source"].value_counts().rename_axis("source").reset_index(name="count"))
print()
print("=== category 분포 (raw) ===")
display(df.groupby(["source", "category"]).size().reset_index(name="count"))

## 2. 카테고리 룰표 v2 적용

raw `category` → `s_category` 매핑.
중복 카테고리 표기(`참여･기반,참여･기반`)는 룰표에 명시된 매핑 사용.

In [ ]:
cat_rule = pd.read_csv(CAT_RULE_CSV, comment="#", dtype=str).fillna("")
print(f"카테고리 룰표 shape: {cat_rule.shape}")
display(cat_rule)

# 룰표를 dict로 변환: (source, raw_category) → s_category
rule_map = {
    (row["source"], row["raw_category"]): row["s_category"]
    for _, row in cat_rule.iterrows()
}

# 적용
def apply_cat_rule(row):
    key = (row["source"], row["category"])
    return rule_map.get(key, "기타")  # 룰표에 없으면 기타로 fallback

df["s_category"] = df.apply(apply_cat_rule, axis=1)

# 미매핑 케이스 중 중복 카테고리("A,A" 형태) 처리
# ex) "참여･기반,참여･기반" → "참여･기반"으로 정규화 후 재시도
unmapped_mask = df["s_category"] == "기타"
for idx in df[unmapped_mask].index:
    raw_cat = df.loc[idx, "category"]
    if "," in raw_cat:
        # 중복 제거: "A,A" → "A"
        parts = [p.strip() for p in raw_cat.split(",")]
        deduped = parts[0] if len(set(parts)) == 1 else raw_cat
        key = (df.loc[idx, "source"], deduped)
        if key in rule_map:
            df.loc[idx, "s_category"] = rule_map[key]

# 최종 미매핑 확인
unmapped = df[df["s_category"] == "기타"]
unmapped_sources = unmapped.groupby(["source", "category"]).size().reset_index(name="count")
unmapped_sources = unmapped_sources[unmapped_sources["count"] > 0]

print()
print(f"=== 룰표 적용 결과: 매핑된 {(df['s_category']!='기타').sum()}건 / 미매핑 {len(unmapped)}건 ===")
if len(unmapped):
    print("\n룰표에 없는 (source, category) 조합:")
    display(unmapped_sources)

print()
print("=== s_category × source 교차표 ===")
display(pd.crosstab(df["s_category"], df["source"], margins=True, margins_name="합계"))

## 3. norm_* 컬럼 생성 (dedupe용)

비교 정확도를 위해 공백/특수문자 제거한 norm 버전 생성.

In [ ]:
_RX_NON = re.compile(r"[\s\W_]+", flags=re.UNICODE)

def normalize_text(s):
    """공백/특수문자 제거 + 소문자"""
    if not s or pd.isna(s):
        return ""
    return _RX_NON.sub("", str(s)).lower()

def normalize_period(start, end):
    """기간 정규화: YYYY-MM-DD~YYYY-MM-DD"""
    s = (start or "").strip()
    e = (end or "").strip()
    if not s and not e:
        return ""
    return f"{s}~{e}"

df["norm_title"]    = df["title"].apply(normalize_text)
df["norm_provider"] = df["provider"].apply(normalize_text)
df["norm_period"]   = df.apply(lambda r: normalize_period(r["apply_start"], r["apply_end"]), axis=1)
df["norm_detail_url"] = df["detail_url"].apply(lambda v: str(v).strip().lower())

# dedupe key: 약한 신호 (제목+기관+기간)
df["_dedupe_key"] = df["norm_title"] + "|" + df["norm_provider"] + "|" + df["norm_period"]

print(f"shape: {df.shape}")
empty_str = ""
print(f"norm_title 빈 값: {(df['norm_title']==empty_str).sum()}")
print(f"norm_detail_url 빈 값: {(df['norm_detail_url']==empty_str).sum()}")

## 3-1. target_age 정규화 → target_age_min / target_age_max

셀프쿼리에서 "25살 청년" 같은 연령 조건을 필터로 활용하려면
`target_age` 문자열을 숫자 컬럼으로 분리해야 합니다.

### 파싱 전략

| 원본 패턴 | min | max |
| :--- | :--- | :--- |
| `만 19세 ~ 만 39세` | 19 | 39 |
| `만 20세 이상 ~ 만 39세 이하` | 20 | 39 |
| `만 20세 미만,만 20세 이상 ~ 만 39세 이하,...` | 첫 번째 유효 범위 사용 | |
| `만 0세 ~ 만 0세` | None | None (0은 무의미) |
| `만 1세 ~ 만 99세` | None | None (사실상 제한 없음) |
| 빈 값 | None | None |

### None 처리 기준
- min=0 또는 max=0 → None (API가 기본값으로 0 넣는 경우)
- min=1, max=99 → None (사실상 제한 없음)
- 빈 값 → None (정보 없음)

In [ ]:
import re as _re

_AGE_RX = _re.compile(
    r'만\s*(\d+)세\s*(?:이상|~)?\s*(?:~\s*)?만?\s*(\d+)세\s*(?:이하)?',
    flags=_re.UNICODE
)

# 전연령대 패턴 (API 기본값 - 사실상 제한 없음)
_FULL_AGE_PATTERNS = [
    "만 20세 미만,만 20세 이상 ~ 만 39세 이하,만 40세 이상",
    "만 20세 이상 ~ 만 39세 이하,만 40세 이상",
    "만 20세 미만,만 20세 이상 ~ 만 39세 이하",
]

# 단방향 나이 패턴 (min만 또는 max만 있는 경우)
_AGE_MIN_ONLY_RX = re.compile(r"^만\s*(\d+)세\s*이상$", flags=re.UNICODE)  # "만 40세 이상"
_AGE_MAX_ONLY_RX = re.compile(r"^만\s*(\d+)세\s*미만$", flags=re.UNICODE)  # "만 19세 미만"

def parse_age_range(raw: str):
    """
    target_age 문자열 → (min_age, max_age) 정수 튜플.
    파싱 불가 / 무의미한 값 / 전연령대 → (None, None)
    """
    if not raw or not raw.strip():
        return None, None

    # 전연령대 패턴 체크 (API 기본값) - 사실상 제한 없음 → NULL
    raw_stripped = raw.strip()
    for pattern in _FULL_AGE_PATTERNS:
        if pattern in raw_stripped:
            return None, None

    # 단방향 패턴 체크 (min만 또는 max만 있는 경우)
    raw_stripped = raw.strip()
    m_min = _AGE_MIN_ONLY_RX.match(raw_stripped)  # "만 40세 이상"
    if m_min:
        return int(m_min.group(1)), None

    m_max = _AGE_MAX_ONLY_RX.match(raw_stripped)  # "만 19세 미만"
    if m_max:
        return None, int(m_max.group(1)) - 1  # "미만"이므로 -1

    # 콤마로 여러 범위가 연결된 경우 → 첫 번째 유효 범위 사용
    candidates = raw.split(',')
    for part in candidates:
        m = _AGE_RX.search(part)
        if m:
            lo, hi = int(m.group(1)), int(m.group(2))
            # 0값은 무의미 (API 기본값)
            if lo == 0 and hi == 0:
                continue
            if lo == 0 or lo == 1:
                lo = None
            if hi == 0 or hi >= 99:
                hi = None
            # lo, hi 둘 다 None이면 제한 없음
            if lo is None and hi is None:
                continue
            return lo, hi

    return None, None

# 적용
age_parsed = df['target_age'].apply(parse_age_range)
df['target_age_min'] = age_parsed.apply(lambda x: x[0]).astype('Int64')
df['target_age_max'] = age_parsed.apply(lambda x: x[1]).astype('Int64')

# 검증
print(f"target_age_min 유효값: {df['target_age_min'].notna().sum()}건")
print(f"target_age_max 유효값: {df['target_age_max'].notna().sum()}건")
print()
print("=== 파싱 결과 샘플 (원본 → min / max) ===")
sample_cols = ['source', 'target_age', 'target_age_min', 'target_age_max', 'title']
check = df[df['target_age'].str.len() > 0][sample_cols].drop_duplicates('target_age').head(15)
display(check)

# 이상값 확인
print()
print("=== 파싱 실패 케이스 (target_age 있는데 min/max 모두 None) ===")
fail = df[
    (df['target_age'].str.len() > 0) &
    (df['target_age_min'].isna()) &
    (df['target_age_max'].isna())
]
print(f"{len(fail)}건")
if len(fail):
    display(fail[['target_age']].value_counts().head(10))


## 4. 스코프 룰표 v2 적용

순서:
1. 모든 행 `_scope = main`, `_scope_reason = primary`로 초기화
2. **카테고리 룰** (Priority 1): `s_category=참여/기반` → other
3. **키워드 룰** (Priority 2): 위원회/협의체 등 보강 키워드
4. **dedupe**:
   - 강한 신호 (URL exact) → other (`duplicate_url`)
   - 완전 동일 (제목+기관+기간 일치) → other (`duplicate_exact`)
   - 자매 공고 → review 큐 (scope는 main 유지)

In [ ]:
# 1. 초기화
df["_scope"] = "main"
df["_scope_reason"] = "primary"

# 2. 룰표 로드
scope_rule = pd.read_csv(SCOPE_RULE_CSV, comment="#", dtype=str).fillna("")
scope_rule["priority"] = pd.to_numeric(scope_rule["priority"], errors="coerce").fillna(99).astype(int)
scope_rule = scope_rule.sort_values("priority")
print(f"스코프 룰표 shape: {scope_rule.shape}")
display(scope_rule[["rule_id", "scope", "scope_reason", "match_field", "match_type", "pattern", "priority"]])

# 3. 룰 적용
def apply_scope_rule(row):
    """룰을 우선순위 순으로 적용. 첫 매칭에서 결정."""
    for _, rule in scope_rule.iterrows():
        match_field = rule["match_field"]
        match_type = rule["match_type"]
        pattern = rule["pattern"]
        
        if match_field not in row.index:
            continue
        
        target_value = str(row[match_field])
        matched = False
        
        if match_type == "category":
            matched = (target_value == pattern)
        elif match_type == "keyword":
            matched = (pattern in target_value)
        elif match_type == "regex":
            try:
                matched = bool(re.search(pattern, target_value))
            except re.error:
                matched = False
        
        if matched:
            return rule["scope"], rule["scope_reason"]
    
    return "main", "primary"

results = df.apply(apply_scope_rule, axis=1)
df["_scope"] = results.apply(lambda x: x[0])
df["_scope_reason"] = results.apply(lambda x: x[1])

print()
print("=== 룰 적용 후 _scope 분포 ===")
display(df["_scope"].value_counts().rename_axis("scope").reset_index(name="count"))
print()
print("=== _scope_reason 분포 ===")
display(df["_scope_reason"].value_counts().rename_axis("reason").reset_index(name="count"))

## 5. Dedupe 자동 처리

### 강한 신호 (URL exact)
같은 detail_url을 가진 행이 2개 이상이면 → 첫 번째만 main 유지, 나머지 other.

### 완전 동일 (제목+기관+기간)
norm_title + norm_provider + norm_period가 100% 일치하면 → 자동 merge (위와 동일 처리).

### 자매 공고 (애매)
_dedupe_key가 같지만 title이 다른 경우 (예: 파리/도쿄 팝업스토어).
→ scope는 main 유지, `_dup_candidate`에 표시 + review 큐로 분리.

In [ ]:
main_only_idx = df[df["_scope"] == "main"].index.tolist()
df["_dup_candidate"] = ""

# === 5.1 강한 신호: URL exact + 제목 동일 ===
# URL만 같고 제목이 다른 경우는 메인페이지 URL 공유 케이스 → 처리 안 함
url_groups = df.loc[main_only_idx][df.loc[main_only_idx, "norm_detail_url"].str.len() > 0].groupby("norm_detail_url")
url_dup_count = 0
url_diff_title_count = 0
for url, g in url_groups:
    if len(g) <= 1:
        continue
    titles = g["norm_title"].unique()
    if len(titles) == 1:
        # URL 같고 제목도 같음 → 진짜 중복 → 자동 처리
        keep_idx = g.index[0]
        drop_idx = g.index[1:]
        df.loc[drop_idx, "_scope"] = "other"
        df.loc[drop_idx, "_scope_reason"] = "duplicate_url"
        df.loc[drop_idx, "_dup_candidate"] = df.loc[keep_idx, "source_id"]
        url_dup_count += len(drop_idx)
    else:
        # URL 같지만 제목 다름 → 메인페이지 URL 공유 케이스 → 처리 안 함
        url_diff_title_count += len(g)
print(f"강한 신호(URL+제목 일치): {url_dup_count}건 자동 처리")
print(f"URL 같지만 제목 다름 (메인페이지 공유): {url_diff_title_count}건 → 처리 안 함")

# === 5.2 완전 동일: norm_title + norm_provider + norm_period 100% 일치 ===
# (URL이 비어있어서 강한 신호로 못 잡힌 케이스 추가 처리)
main_idx_after_url = df[df["_scope"] == "main"].index.tolist()
exact_groups = df.loc[main_idx_after_url].groupby("_dedupe_key")
exact_dup_count = 0
for key, g in exact_groups:
    if len(g) <= 1:
        continue
    if not key or "||" in key:  # 빈 키 또는 부분 빈 키 제외
        continue
    
    # 그룹 내 norm_title이 모두 같으면 → 완전 동일 (자동 merge)
    titles = g["norm_title"].unique()
    if len(titles) == 1 and titles[0]:
        keep_idx = g.index[0]
        drop_idx = g.index[1:]
        df.loc[drop_idx, "_scope"] = "other"
        df.loc[drop_idx, "_scope_reason"] = "duplicate_exact"
        df.loc[drop_idx, "_dup_candidate"] = df.loc[keep_idx, "source_id"]
        exact_dup_count += len(drop_idx)
print(f"완전 동일 자동 처리: {exact_dup_count}건")

# === 5.3 자매 공고: _dedupe_key 같은데 title 다름 → review 큐 ===
main_idx_now = df[df["_scope"] == "main"].index.tolist()
sister_groups = df.loc[main_idx_now].groupby("_dedupe_key")
sister_review_pairs = []
for key, g in sister_groups:
    if len(g) <= 1 or not key or "||" in key:
        continue
    sister_review_pairs.append((key, g.index.tolist()))

sister_count = sum(len(idx_list) for _, idx_list in sister_review_pairs)
print(f"자매 공고 (review 큐): {len(sister_review_pairs)}그룹 / {sister_count}건")

print()
print("=== dedupe 후 최종 _scope 분포 ===")
display(df["_scope"].value_counts().rename_axis("scope").reset_index(name="count"))
print()
print("=== _scope_reason 분포 ===")
display(df["_scope_reason"].value_counts().rename_axis("reason").reset_index(name="count"))

## 6. Review 큐 생성

자매 공고 그룹을 검토자가 엑셀로 열어서 결정할 수 있도록 CSV로 분리.

### 검토자가 채울 컬럼
- `review_decision`: `merge` / `keep_separate` / `unsure`
- `review_canonical`: merge일 경우 어느 source_id를 대표로 할지
- `review_note`: 자유 메모

In [ ]:
review_rows = []
for review_idx, (key, idx_list) in enumerate(sister_review_pairs, start=1):
    group = df.loc[idx_list]
    for _, row in group.iterrows():
        review_rows.append({
            "review_id": f"REV{review_idx:04d}",
            "group_size": len(group),
            "source": row["source"],
            "source_id": row["source_id"],
            "title": row["title"],
            "provider": row["provider"],
            "apply_start": row["apply_start"],
            "apply_end": row["apply_end"],
            "norm_title": row["norm_title"],
            "norm_provider": row["norm_provider"],
            "norm_period": row["norm_period"],
            "_dedupe_key": row["_dedupe_key"],
            # 검토자가 채울 컬럼
            "review_decision": "",  # merge / keep_separate / unsure
            "review_canonical": "",
            "review_note": "",
        })

review_df = pd.DataFrame(review_rows)

if len(review_df):
    review_df.to_csv(REVIEW_CSV, index=False, encoding="utf-8-sig")
    print(f"review 큐 저장: {REVIEW_CSV}")
    print(f"총 {len(review_df)}건 ({len(sister_review_pairs)}개 그룹)")
    
    print()
    print("=== review 큐 미리보기 (전체 그룹) ===")
    for rid in review_df["review_id"].unique():
        g = review_df[review_df["review_id"] == rid]
        print(f"\n[{rid}] group_size={g['group_size'].iloc[0]}")
        display(g[["source", "source_id", "title", "provider"]])
else:
    print("자매 공고 그룹 없음. review 큐 생성 안 함.")

## 6-1. target_tags 컬럼 생성

target_tags_rule.csv 룰표를 적용해 공고 대상자 태그를 자동 부여합니다.

### 태그 11종
- 청년 / 예비창업자 / 초기창업기업 / 중소기업 / 소상공인 / 스타트업
- 사회적기업 / 기업 / 농업인 / 여성 / 누구나

### 적용 방식
- 우선순위(priority) 순으로 룰 적용
- 복수 태그 허용
- 매칭 안 되면 기타로 분류

In [ ]:
tags_rule = pd.read_csv(TAGS_RULE_CSV, comment="#", dtype=str).fillna("")
tags_rule["priority"] = pd.to_numeric(tags_rule["priority"], errors="coerce").fillna(99).astype(int)
tags_rule = tags_rule.sort_values("priority")
print(f"target_tags 룰표 shape: {tags_rule.shape}")

def apply_target_tags(row, rules):
    tags = set()
    for _, rule in rules.iterrows():
        field = rule["match_field"]
        mtype = rule["match_type"]
        pattern = rule["pattern"]
        tag = rule["tag"]
        
        if field not in row.index:
            continue
        val = str(row[field])
        
        if mtype == "keyword" and pattern in val:
            tags.add(tag)
        elif mtype == "category" and val == pattern:
            tags.add(tag)
        elif mtype == "code" and pattern in val:
            tags.add(tag)
    
    return ",".join(sorted(tags)) if tags else "기타"

df["target_tags"] = df.apply(lambda r: apply_target_tags(r, tags_rule), axis=1)

# 분포 확인
from collections import Counter
tag_counter = Counter()
for tags in df["target_tags"]:
    for t in tags.split(","):
        tag_counter[t.strip()] += 1

print()
print("=== target_tags 분포 ===")
tag_dist = pd.DataFrame(
    [(tag, cnt) for tag, cnt in sorted(tag_counter.items(), key=lambda x: -x[1])],
    columns=["tag", "count"]
)
display(tag_dist)

print()
etc_count = (df["target_tags"] == "기타").sum()
total = len(df)
coverage = (total - etc_count) / total * 100
print(f"태그 없음(기타): {etc_count}건")
print(f"커버리지: {coverage:.1f}%")

## 6-2. region 정규화 (룰표 `region_rule.csv` 적용)

`region` 컬럼에 잘못 들어간 값들(소관부처명, 기관명, 짧은 표기 alias)을 룰표로 일괄 정리합니다.

### 정리 대상
- **biz**: `jrsdInsttNm`(소관부처명)이 region에 들어간 케이스 → "전국"으로 매핑 (예: "중소벤처기업부" → "전국")
- **youth**: `rgtrInstCdNm`(등록기관명)이 region에 들어간 케이스 → 시도 prefix 추출 (예: "제주특별자치도경제통상진흥원 경영본부 청년센터" → "제주특별자치도")
- **kst**: 짧은 표기 → 긴 공식 표기 통일 (예: "서울" → "서울특별시")

### 룰표 형식 (`data/csv/rule/region_rule.csv`)
- `match_type`:
  - `exact`: region 값이 pattern과 **완전히 같은지**
  - `prefix`: region 값이 pattern**으로 시작**하는지
  - `contains`: region 값에 pattern이 **포함**되어 있는지
- `priority`: 숫자가 작을수록 먼저 적용. 첫 매칭에서 결정 종료.

### 적용 시점
모든 컬럼 처리(카테고리/스코프/dedupe/target_tags)가 끝난 뒤, main_csv 저장 직전.

In [ ]:
# 1. 룰표 로드 + priority 정렬
region_rule = pd.read_csv(REGION_RULE_CSV, comment="#", dtype=str).fillna("")
region_rule["priority"] = pd.to_numeric(region_rule["priority"], errors="coerce").fillna(99).astype(int)
region_rule = region_rule.sort_values("priority")
print(f"region 룰표 shape: {region_rule.shape}")

# 2. 제목 bracket prefix 추출 함수
# "[강원] ...", "[충북] ..." 처럼 제목 앞에 대괄호로 감싼 지역명 추출
import re as _re_region
_BRACKET_RX = _re_region.compile(r"^\[([^\]]+)\]")

# bracket keyword → 광역시도 매핑
_BRACKET_MAP = {
    "서울": "서울특별시",
    "부산": "부산광역시",
    "대구": "대구광역시",
    "인천": "인천광역시",
    "광주": "광주광역시",
    "대전": "대전광역시",
    "울산": "울산광역시",
    "세종": "세종특별자치시",
    "경기": "경기도",
    "강원": "강원특별자치도",
    "충북": "충청북도",
    "충남": "충청남도",
    "전북": "전북특별자치도",
    "전남": "전라남도",
    "경북": "경상북도",
    "경남": "경상남도",
    "제주": "제주특별자치도",
    "대구ㆍ경북": "경상북도",
    "부산ㆍ울산ㆍ경남": "경상남도",
}

# 시 단위 → 시도 매핑 (title 키워드 기반)
_CITY_MAP = {
    # 충청남도
    "천안": "충청남도", "아산": "충청남도",
    # 강원특별자치도
    "춘천": "강원특별자치도", "원주": "강원특별자치도",
    "강릉": "강원특별자치도", "동해": "강원특별자치도",
    # 경상북도
    "경산": "경상북도", "포항": "경상북도",
    "안동": "경상북도", "구미": "경상북도",
    # 경기도
    "광명": "경기도", "시흥": "경기도",
    "수원": "경기도", "성남": "경기도",
    # 전라남도
    "나주": "전라남도", "순천": "전라남도",
    # 전북특별자치도
    "전주": "전북특별자치도", "군산": "전북특별자치도",
    # 충청북도
    "충주": "충청북도", "청주": "충청북도",
    # 경상남도
    "진주": "경상남도", "창원": "경상남도",
}

def extract_title_region(title):
    """제목 앞의 [지역] prefix 추출 → 광역시도명 반환. 없으면 None."""
    m = _BRACKET_RX.match(str(title))
    if not m:
        return None
    prefix = m.group(1).strip()
    # 정확히 매핑되는 키 먼저
    if prefix in _BRACKET_MAP:
        return _BRACKET_MAP[prefix]
    # 부분 매칭 (예: "강원" → "강원특별자치도")
    for key, val in _BRACKET_MAP.items():
        if key in prefix:
            return val
    return None

def extract_city_region(title):
    """title에 시 단위 지명이 있으면 시도 반환. 없으면 None."""
    for city, sido in _CITY_MAP.items():
        if city in str(title):
            return sido
    return None

# 3. 룰 적용 함수 (region 컬럼 기준)
def apply_region_rule(region):
    if not region or pd.isna(region):
        return region
    region = str(region).strip()
    for _, rule in region_rule.iterrows():
        if rule["match_type"] in ("title_bracket",):
            continue  # title_bracket은 별도 처리
        match_type = rule["match_type"]
        pattern = rule["pattern"]
        replacement = rule["replacement"]
        matched = False
        if match_type == "exact":
            matched = (region == pattern)
        elif match_type == "prefix":
            matched = region.startswith(pattern)
        elif match_type == "contains":
            matched = (pattern in region)
        if matched:
            return replacement
    return region

# 4. 적용 (변경 전 값을 _region_before에 임시 보관)
df["_region_before"] = df["region"]

# Step 1: 제목 prefix 최우선 적용
title_region_applied = 0
for idx, row in df.iterrows():
    title_region = extract_title_region(row["title"])
    if title_region:
        df.at[idx, "region"] = title_region
        title_region_applied += 1

print(f"[title bracket] 적용: {title_region_applied}건")

# Step 2: region 룰표 적용 (title prefix로 이미 바뀐 건 유지)
df["region"] = df["region"].apply(apply_region_rule)

# Step 3: title 시 단위 지명 기반 보정
# 룰표 적용 후에도 전국인 경우만 보정 (부처명→전국 변환된 케이스 포함)
city_region_applied = 0
for idx, row in df.iterrows():
    if row["region"] == "전국":
        city_region = extract_city_region(row["title"])
        if city_region:
            df.at[idx, "region"] = city_region
            city_region_applied += 1
print(f"[title city] 적용: {city_region_applied}건")

# 5. 변경된 케이스 출력
changed = df[df["_region_before"] != df["region"]]
print(f"\n=== region 변경된 행: {len(changed)}건 ===")
display(
    changed.groupby(["_region_before", "region"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

# 6. 정규화 후 region 분포
print("\n=== 정규화 후 region 분포 (top 25) ===")
display(df["region"].value_counts().head(25).rename_axis("region").reset_index(name="count"))

# 7. 어떤 룰에도 안 걸린 region (의심 케이스)
known_regions = {
    "전국", "서울특별시", "부산광역시", "대구광역시", "인천광역시",
    "광주광역시", "대전광역시", "울산광역시", "세종특별자치시",
    "경기도", "강원특별자치도", "충청북도", "충청남도",
    "전북특별자치도", "전라남도", "경상북도", "경상남도", "제주특별자치도", ""
}
suspicious = df[~df["region"].isin(known_regions)]
if len(suspicious):
    print(f"\n=== 정규화 후 의심 케이스: {len(suspicious)}건 ===")
    display(suspicious[["source", "region", "title"]].head(10))
else:
    print("\n✅ 정규화 후 의심 케이스 없음")

## 7. 임베딩용 main_csv 저장

`_scope=main`인 행만 추려서 임베딩 단계로 넘길 수 있는 깨끗한 CSV 생성.

### 컬럼 구성
- 식별: `source`, `source_id`
- 표시용: `title`, `summary`, `s_category`, `provider`, `region`
- raw 메타: `target_group`, `target_age`, `support_type`, `apply_start`, `apply_end`, `detail_url`
- 스코프: `_scope`, `_scope_reason`
- 정규화: `norm_title`, `norm_provider`, `norm_period`

In [ ]:
main_df = df[df["_scope"] == "main"].copy()

MAIN_COLS = [
    # 식별
    "source", "source_id",
    # 서비스 표시용 (임베딩 대상)
    "title", "summary", "s_category", "provider", "region",
    # raw 메타 (셀프쿼리용 메타데이터)
    "target_group", "target_tags", "target_age", "target_age_min", "target_age_max", "target_detail",
    "income_condition", "startup_stage", "support_type",
    "apply_start", "apply_end",
    "additional_conditions", "required_documents", "application_method",
    "detail_url",
    # 스코프 정보
    "_scope", "_scope_reason",
    # 정규화 (dedupe 추적용)
    "norm_title", "norm_provider", "norm_period",
    # 원본 카테고리 (룰표 변경 시 재적용 가능하도록 보존)
    "category", "subcategory",
]

# 누락 컬럼은 빈 문자열로
for col in MAIN_COLS:
    if col not in main_df.columns:
        main_df[col] = ""

main_out = main_df[MAIN_COLS].copy()
main_out.to_csv(MAIN_CSV, index=False, encoding="utf-8-sig")

print(f"main 저장: {MAIN_CSV}")
print(f"shape: {main_out.shape}")
print()
print("=== 최종 s_category × source (main만) ===")
display(pd.crosstab(main_out["s_category"], main_out["source"], margins=True, margins_name="합계"))
print()
print("=== 미리보기 ===")
display(main_out.head(3))

## 8. 요약 + 다음 단계

### 처리 결과 요약

In [ ]:
total = len(df)
main_count = (df["_scope"] == "main").sum()
other_count = (df["_scope"] == "other").sum()

reason_counts = df["_scope_reason"].value_counts()

print("=" * 60)
print("PolicyRec v1.1.10 처리 결과")
print("=" * 60)
print(f"총 raw 데이터: {total}건")
print(f"  → main (임베딩 대상): {main_count}건")
print(f"  → other: {other_count}건")
print()
print("[other 상세]")
for reason, cnt in reason_counts.items():
    if reason != "primary":
        print(f"  - {reason}: {cnt}건")
print()
print(f"review 큐: {len(sister_review_pairs)}그룹 / {sister_count}건")
print()
print("=" * 60)
print("다음 단계")
print("=" * 60)
print(f"1. 임베딩 노트북에서 {MAIN_CSV.name} 입력으로 사용")
print(f"   - 임베딩 대상 텍스트: title + summary + s_category + region")
print(f"   - 셀프쿼리 메타데이터: s_category, region, target_age, target_group 등")
print(f"2. (선택) review 큐 검토 후 추가 dedupe 반영")
print()
print("=" * 60)
print("v1.1.9 → v1.1.10 변경사항")
print("=" * 60)
print("- region 정규화 룰표(region_rule.csv) 신규 적용")
print("  · 부처명/기관명 혼입 보정 (biz jrsdInsttNm, youth rgtrInstCdNm)")
print("  · 시도 alias 통일 (서울 → 서울특별시 등)")
print("  · 변경 분포는 위 6-2 섹션 출력 참고")
print("- 룰표 파일명 정책: 단일 파일 패턴 도입")
print("  · region_rule.csv: 버전 미부여 (target_tags_rule.csv 패턴 따름)")
print("  · cat_rule, scope_rule: v1.1.8 파일 그대로 참조 (룰 변경 없음)")

## 9. 임베딩 단계 인수인계

### 임베딩에 사용할 텍스트 컬럼
다음 컬럼을 조합해서 임베딩 텍스트 생성 권장:

```python
def build_embedding_text(row):
    parts = [
        f"제목: {row['title']}",
        f"카테고리: {row['s_category']}",
        f"지역: {row['region']}",
        f"대상: {row['target_group']}",
        f"요약: {row['summary']}",
    ]
    return "\n".join(parts)
```

### 셀프쿼리용 메타데이터 컬럼
Supabase 적재 시 별도 컬럼으로 보관 (필터 쿼리에 사용):
- `s_category`: 카테고리 필터
- `region`: 지역 필터
- `target_age`: 연령 필터
- `target_group`: 대상자 필터
- `apply_start`, `apply_end`: 모집 기간 필터
- `_scope_reason`: 디버깅/통계용

### 룰표 수정 흐름
1. `cat_rule.csv` 또는 `scope_rule.csv` 수정
2. 이 노트북 재실행
3. main_csv 새로 생성 → 임베딩 노트북도 재실행